In [45]:
import os, io
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import gzip
import requests

import tensorflow as tf

from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.layers import Conv2D, ReLU, MaxPooling2D, UpSampling2D, Dropout, BatchNormalization, Flatten, Dense, Conv2DTranspose, GlobalAveragePooling2D, DepthwiseConv2D
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [46]:
def get_dataset(train_generator, batch_size=32, val_split=0.2, random_state=42, option='MNIST'):
    data = ''
    if(option == 'MNIST'):
        data = tf.keras.datasets.mnist
    elif(option == 'fashionMNIST'):
        data = tf.keras.datasets.fashion_mnist
    elif(option == 'cifar10'):
        data = tf.keras.datasets.cifar10

    (x_train, y_train), (x_test, y_test) = data.load_data()

    if option != 'cifar10':
        x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
        x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0
    else:
        x_train = x_train.astype("float32") / 255.0
        x_test = x_test.astype("float32") / 255.0
    #Split data to train and valitation
    x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size = val_split, random_state=random_state)

    num_classes = 10  # MNIST has 10 classes (digits 0-9)
    y_train = to_categorical(y_train, num_classes)
    y_val = to_categorical(y_val, num_classes)
    y_test = to_categorical(y_test, num_classes)

    #Use default ImageDataGenerator for inmutable images
    val_test_generator = ImageDataGenerator()
    train_gen = train_generator.flow(x_train, y_train, batch_size=batch_size)
    val_gen = val_test_generator.flow(x_val, y_val, batch_size=batch_size)
    test_gen = val_test_generator.flow(x_test, y_test, batch_size=batch_size, shuffle=False)

    return train_gen, val_gen, test_gen, y_test

In [47]:
train_generator = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.2,
    zoom_range=0.2)

train_data_mnist, val_data_mnist, test_data_mnist, y_true_mnist = get_dataset(train_generator, batch_size=64, val_split=0.2, random_state=42, option='MNIST')


In [48]:
def create_model(option='MNIST'):
    model = keras.Sequential()
    model.add(Conv2D(16, (5, 5), strides=(1, 1), activation=None, padding='same', input_shape=(28, 28, 1)))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(32, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(64, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(64, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(64, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(GlobalAveragePooling2D())
    model.add(Dropout(0.3))
    model.add(Dense(10, activation='softmax'))

    optimizer = Adam(learning_rate=2e-3)
    loss_fn = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=optimizer,
                  loss=loss_fn,
                  metrics=['accuracy', 'Precision', 'Recall'])
    model.summary()

    return model

model_mnist = create_model(option='MNIST')

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_25 (Conv2D)                   │ (None, 28, 28, 16)          │             416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_75               │ (None, 28, 28, 16)          │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_75 (ReLU)                      │ (None, 28, 28, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ depthwise_conv2d_50                  │ (None, 28, 28, 16)          │             160 │
│ (DepthwiseConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_76               │ (None, 28, 28, 16)          │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_76 (ReLU)                      │ (None, 28, 28, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ depthwise_conv2d_51                  │ (None, 28, 28, 16)          │             160 │
│ (DepthwiseConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_77               │ (None, 28, 28, 16)          │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_77 (ReLU)                      │ (None, 28, 28, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_26 (Conv2D)                   │ (None, 28, 28, 32)          │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_78               │ (None, 28, 28, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_78 (ReLU)                      │ (None, 28, 28, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_20 (MaxPooling2D)      │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_25 (Dropout)                 │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ depthwise_conv2d_52                  │ (None, 14, 14, 32)          │             320 │
│ (DepthwiseConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_79               │ (None, 14, 14, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_79 (ReLU)                      │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 105,738 (413.04 KB)

 Trainable params: 104,298 (407.41 KB)

 Non-trainable params: 1,440 (5.62 KB)

In [49]:
lr_scheduler_mnist = ReduceLROnPlateau(monitor='loss', patience=3, factor=0.5, min_lr=5e-5)

In [50]:
history_mnist = model_mnist.fit(train_data_mnist,
         validation_data=val_data_mnist,
         epochs=80,
         batch_size=64,
         shuffle=True
         ,callbacks=[lr_scheduler_mnist])

Epoch 1/80
750/750 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - Precision: 0.8577 - Recall: 0.3569 - accuracy: 0.5566 - loss: 1.5259 - val_Precision: 0.9842 - val_Recall: 0.9570 - val_accuracy: 0.9740 - val_loss: 0.5972 - learning_rate: 0.0020
Epoch 2/80
750/750 ━━━━━━━━━━━━━━━━━━━━ 20s 27ms/step - Precision: 0.9662 - Recall: 0.8815 - accuracy: 0.9358 - loss: 0.7481 - val_Precision: 0.9793 - val_Recall: 0.9523 - val_accuracy: 0.9680 - val_loss: 0.6162 - learning_rate: 0.0020
Epoch 3/80
750/750 ━━━━━━━━━━━━━━━━━━━━ 19s 25ms/step - Precision: 0.9762 - Recall: 0.9116 - accuracy: 0.9526 - loss: 0.6999 - val_Precision: 0.9865 - val_Recall: 0.9709 - val_accuracy: 0.9786 - val_loss: 0.5971 - learning_rate: 0.0020
Epoch 4/80
750/750 ━━━━━━━━━━━━━━━━━━━━ 20s 26ms/step - Precision: 0.9814 - Recall: 0.9276 - accuracy: 0.9635 - loss: 0.6738 - val_Precision: 0.9904 - val_Recall: 0.9835 - val_accuracy: 0.9862 - val_loss: 0.5664 - learning_rate: 0.0020
Epoch 5/80
750/750 ━━━━━━━━━━━━━━━━━━━━ 19s 25ms/step - 

In [51]:
model_mnist.save('models/model_mnist.h5')
model_mnist.save('models/model_mnist.keras')

In [52]:
def get_model_metrics(model, test_data, model_title='model'):
    res = model.evaluate(test_data)
    loss, acc, prec, rec = res
    print(model_title)
    print(f'The accuracy of the model is {acc}')
    print(f'The loss of the model is {loss}')
    print(f'The precision of the model is {prec}')
    print(f'The recall of the model is {rec}')
    print('-' * 150 + '\n')

get_model_metrics(model_mnist, test_data_mnist,model_title='Model metrics for MNIST dataset')

 24/157 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - Precision: 0.9981 - Recall: 0.9953 - accuracy: 0.9959 - loss: 0.5382

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - Precision: 0.9970 - Recall: 0.9944 - accuracy: 0.9950 - loss: 0.5413
Model metrics for MNIST dataset
The accuracy of the model is 0.9962999820709229
The loss of the model is 0.5401418209075928
The precision of the model is 0.9975951910018921
The recall of the model is 0.9955999851226807
------------------------------------------------------------------------------------------------------------------------------------------------------

